# Phase 1: Data and baselines

Load BTC-USD daily data, time-based train/val/test split, two baselines (last value, moving average), and report MAE, RMSE, directional accuracy.

In [3]:
# Colab: clone repo and install deps if not already there. Local: just pip install.
import subprocess
import sys
from pathlib import Path

if 'google.colab' in sys.modules:
    repo_dir = Path('/content/crypto-price-prediction')
    if not (repo_dir / 'src').exists():
        subprocess.run(['git', 'clone', '-q', 'https://github.com/MOONx02/crypto-price-prediction.git', '/content/crypto-price-prediction'], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/content/crypto-price-prediction/requirements.txt'], check=True)
else:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'numpy', 'yfinance', 'pyarrow'])

0

In [4]:
import pandas as pd
import numpy as np
import yfinance as yf
from pathlib import Path
import sys

if 'google.colab' in sys.modules:
    ROOT = Path('/content/crypto-price-prediction')
else:
    ROOT = Path('.').resolve()
    if ROOT.name == 'notebooks':
        ROOT = ROOT.parent
    elif ROOT.name != 'crypto-price-prediction':
        ROOT = ROOT / 'crypto-price-prediction'
sys.path.insert(0, str(ROOT))
from src.metrics import regression_metrics

## 1. Download and clean data

In [5]:
TICKER = 'BTC-USD'
START = '2017-01-01'   # ~8 years of daily data; use '2015-01-01' for even more
END = None             # up to now
DATA_DIR = ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)
cache_path = DATA_DIR / f'{TICKER.replace("-", "_")}_daily.parquet'

if cache_path.exists():
    df = pd.read_parquet(cache_path)
    print('Loaded from cache:', cache_path)
else:
    raw = yf.download(TICKER, start=START, end=END, progress=False, auto_adjust=True)
    if raw.index.nlevels > 1:
        raw = raw.reset_index(level=1, drop=True)
    raw.index = pd.to_datetime(raw.index).tz_localize(None)
    raw = raw.sort_index()
    raw = raw.ffill().dropna()
    df = raw[['Close']].copy()
    df.columns = ['price']
    if 'Volume' in raw.columns:
        df['volume'] = raw['Volume']
    df.to_parquet(cache_path)
    print('Downloaded and saved to', cache_path)
print(df.head(), df.shape)

Downloaded and saved to /Users/moon/Library/Mobile Documents/com~apple~CloudDocs/WINTER2025/211/crypto-price-prediction/data/BTC_USD_daily.parquet
                  price     volume
Date                              
2017-01-01   998.325012  147775008
2017-01-02  1021.750000  222184992
2017-01-03  1043.839966  185168000
2017-01-04  1154.729980  344945984
2017-01-05  1013.380005  510199008 (3340, 2)


In [6]:
# Optional: download more tickers (e.g. ETH) into data/ for later use
for ticker in ['ETH-USD']:  # add more: 'ETH-USD', 'SOL-USD', ...
    path = DATA_DIR / f'{ticker.replace("-", "_")}_daily.parquet'
    if path.exists():
        print('Already cached:', path)
        continue
    raw = yf.download(ticker, start=START, end=END, progress=False, auto_adjust=True)
    if raw.empty or len(raw) < 100:
        print('Skipping', ticker, '(insufficient data)')
        continue
    if raw.index.nlevels > 1:
        raw = raw.reset_index(level=1, drop=True)
    raw.index = pd.to_datetime(raw.index).tz_localize(None)
    raw = raw.sort_index().ffill().dropna()
    out = raw[['Close']].copy()
    out.columns = ['price']
    if 'Volume' in raw.columns:
        out['volume'] = raw['Volume']
    out.to_parquet(path)
    print('Downloaded', ticker, '->', path, 'shape', out.shape)

Downloaded ETH-USD -> /Users/moon/Library/Mobile Documents/com~apple~CloudDocs/WINTER2025/211/crypto-price-prediction/data/ETH_USD_daily.parquet shape (3028, 2)


**Data size:** ~3.3k daily rows (2017–now) is **more than enough** for lag-feature models, LSTM, and ablation. Only change `START` to e.g. `'2015-01-01'` if you want extra history for experiments.

## 2. Time-based split (70 / 15 / 15)

In [7]:
n = len(df)
train_end = int(0.70 * n)
val_end = int(0.85 * n)

train = df.iloc[:train_end]
val = df.iloc[train_end:val_end]
test = df.iloc[val_end:]

print('Train', train.index[0], '->', train.index[-1], len(train))
print('Val  ', val.index[0], '->', val.index[-1], len(val))
print('Test ', test.index[0], '->', test.index[-1], len(test))

Train 2017-01-01 00:00:00 -> 2023-05-27 00:00:00 2338
Val   2023-05-28 00:00:00 -> 2024-10-09 00:00:00 501
Test  2024-10-10 00:00:00 -> 2026-02-22 00:00:00 501


## 3. Next-day target and baselines on test set

In [8]:
y_true = test['price'].values[1:]
y_prev = test['price'].values[:-1]

# Baseline 1: last value (predict tomorrow = today)
pred_last = y_prev

# Baseline 2: 7-day moving average (MA of past 7 days predicts next day)
prices = test['price'].values
window = 7
pred_ma = np.array([np.mean(prices[i - window:i]) for i in range(window, len(prices))])
y_true_ma = y_true[window - 1:]  # same length as pred_ma

m_last = regression_metrics(y_true, pred_last)
m_ma = regression_metrics(y_true_ma, pred_ma)

print('Baseline 1 (last value):', m_last)
print('Baseline 2 (7-day MA): ', m_ma)

Baseline 1 (last value): {'mae': 1585.502890625, 'rmse': 2210.140134333367, 'directional_accuracy': 0.0}
Baseline 2 (7-day MA):  {'mae': 2678.2534001771255, 'rmse': 3562.4564689694266, 'directional_accuracy': 0.49290060851926976}


In [9]:
print('Test set metrics')
print('                 MAE       RMSE   Dir.Acc')
print('Last value  ', f"{m_last['mae']:>10.2f}", f"{m_last['rmse']:>10.2f}", f"{m_last['directional_accuracy']:>8.2%}")
print('7-day MA     ', f"{m_ma['mae']:>10.2f}", f"{m_ma['rmse']:>10.2f}", f"{m_ma['directional_accuracy']:>8.2%}")

Test set metrics
                 MAE       RMSE   Dir.Acc
Last value      1585.50    2210.14    0.00%
7-day MA         2678.25    3562.46   49.29%
